# Explanation: This file is used for generating plot data

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt
import os
import math

In [ ]:
# File path settings
base_path = r'..\code\\'
file_path = os.path.join(base_path, 'patent_lcc_dataset.csv')

# Read data
df = pd.read_csv(file_path)
df.head(3)

In [ ]:
# Create new DataFrame to save processing results
df_ln = df[['patent_id']].copy()

# Define threshold constants
THRESHOLD = 0.00005

# Process each column except the first one
for col in df.columns[1:]:
    # Step 1: Replace values less than threshold with threshold
    temp = df[col].mask(df[col] < THRESHOLD, THRESHOLD)
    
    # Step 2: Take natural logarithm
    temp = np.log(temp)
    
    # Step 3: Take absolute value
    temp = np.abs(temp)
    
    # Step 4: Reverse maximum value
    max_val = temp.max()
    temp = max_val - temp
    
    # Add processed columns to df_ln
    df_ln[col] = temp


# Display first few rows of processed data (optional)
print(df_ln.head())

In [ ]:
# Get columns to plot
plot_columns = df_ln.columns[1:]

# Calculate subplot rows and columns
n_cols = 3
n_rows = int(np.ceil(len(plot_columns)/n_cols))

# Create subplot canvas
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

# Iterate to plot each feature
for idx, col in enumerate(plot_columns):
    sns.kdeplot(df_ln[col], ax=axes[idx], fill=True)
    axes[idx].set_title(f'KDE of {col}')
    axes[idx].set_xlabel('Value')
    
# Hide extra subplots
for j in range(len(plot_columns), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Set figure style (seaborn-poster style has larger font size, enhanced here)
# plt.style.use('seaborn-poster')
fig, ax = plt.subplots(figsize=(9, 4))

# Draw KDE curves
for col in df_ln.columns[1:]:
    sns.kdeplot(data=df_ln, x=col, ax=ax, alpha=0.7, linewidth=2.0, label=col)

# Visualization enhancement
#ax.set_title(f"KDE of Transformed Features (Threshold={THRESHOLD})", 
            # pad=20, 
            # fontsize=18)  # Title font size increased to 18
ax.set_xlabel("Normalized Value Range", 
             labelpad=10, 
             fontsize=16)  # X-axis label font size 16
ax.set_ylabel("Probability Density", 
             labelpad=10, 
             fontsize=16)  # Y-axis label font size 16

# Set tick label font sizes
ax.tick_params(axis='both', which='major', labelsize=16)

# Set legend font size
plt.legend(bbox_to_anchor=(0.8, 0.8), 
           borderaxespad=0,
           fontsize=14)  # Legend text font size 12

plt.tight_layout()

# Save as PDF (vector graphics font size does not affect clarity)
plt.savefig("kde_plot.svg", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
df.head(5)

In [ ]:
df_ln.head()

In [ ]:
df_ln.to_csv('patent_score_ln_3d.csv', index=False)

4 Venn Diagram

In [ ]:
# Read data
df = pd.read_csv(r'patent_lcc_dataset.csv')
df = df[['patent_id', 'pr', 'au', 'hub']]

# Define function to get top 10% patents
def get_top_10_patents(series):
    n = math.ceil(len(series) * 0.1)
    return series.head(n).tolist()

# Get top 10% patents for each metric
pr_top = df.sort_values('pr', ascending=False)['patent_id'].pipe(get_top_10_patents)
au_top = df.sort_values('au', ascending=False)['patent_id'].pipe(get_top_10_patents)
hub_top = df.sort_values('hub', ascending=False)['patent_id'].pipe(get_top_10_patents)

# Create dataframe and convert format
venn_data = []
for lst, metric in zip([pr_top, au_top, hub_top], ['pr', 'au', 'hub']):
    venn_data.extend([[f"{pid}", metric] for pid in lst])

venn_df = pd.DataFrame(venn_data, columns=['patent_metric', 'metric'])

# Save results
venn_df.to_csv('venn1.csv', index=False)
print("Processing completed, results saved as venn1.csv")

In [ ]:
# Read data
df = pd.read_csv(r'patent_lcc_dataset.csv')
df = df[['patent_id', 'PR', 'AU', 'HUB']]

# Define function to get top 10% patents
def get_top_10_patents(series):
    n = math.ceil(len(series) * 0.1)
    return series.head(n).tolist()

# Get top 10% patents for each metric
pr_top = df.sort_values('PR', ascending=False)['patent_id'].pipe(get_top_10_patents)
au_top = df.sort_values('AU', ascending=False)['patent_id'].pipe(get_top_10_patents)
hub_top = df.sort_values('HUB', ascending=False)['patent_id'].pipe(get_top_10_patents)

# Create dataframe and convert format
venn_data = []
for lst, metric in zip([pr_top, au_top, hub_top], ['PR', 'AU', 'HUB']):
    venn_data.extend([[f"{pid}", metric] for pid in lst])

venn_df = pd.DataFrame(venn_data, columns=['patent_metric', 'metric'])

# Save results
venn_df.to_csv('venn2.csv', index=False)
print("Processing completed, results saved as venn2.csv")

3D

In [ ]:
# Read data
path = r'patent_score_ln_3d.csv'
df = pd.read_csv(path)

df = df[['patent_id', 'P', 'pr', 'au', 'hub']].copy()


# Initialize count column
df['count'] = 0

# Iterate through each metric for processing
for metric in ['P', 'pr', 'au', 'hub']:
    # Sort by current metric in descending order
    sorted_df = df.sort_values(by=metric, ascending=False)
    
    # Calculate top 10% count (round up)
    n = math.ceil(len(sorted_df) * 0.1)
    
    # Get top n patent IDs
    top_patents = sorted_df['patent_id'].head(n).tolist()
    
    # Increment count for patents in top 10%
    df.loc[df['patent_id'].isin(top_patents), 'count'] += 1

# Filter records with count > 0
filtered_df = df[df['count'] > 0]

# Save filtered results
filtered_df[['patent_id', 'P', 'pr', 'au', 'hub', 'count']].to_csv('count1.csv', index=False)





print(f"Processing completed, {len(filtered_df)} valid records retained")

In [ ]:
# Read data
path = r'patent_score_ln_3d.csv'
df = pd.read_csv(path)

df = df[['patent_id', 'P', 'PR', 'AU', 'HUB']].copy()


# Initialize count column
df['count'] = 0

# Iterate through each metric for processing
for metric in ['P', 'PR', 'AU', 'HUB']:
    # Sort by current metric in descending order
    sorted_df = df.sort_values(by=metric, ascending=False)
    
    # Calculate top 10% count (round up)
    n = math.ceil(len(sorted_df) * 0.1)
    
    # Get top n patent IDs
    top_patents = sorted_df['patent_id'].head(n).tolist()
    
    # Increment count for patents in top 10%
    df.loc[df['patent_id'].isin(top_patents), 'count'] += 1

# Filter records with count > 0
filtered_df = df[df['count'] > 0]

# Save filtered results
filtered_df[['patent_id', 'P', 'PR', 'AU', 'HUB', 'count']].to_csv('count2.csv', index=False)

print(f"Processing completed, {len(filtered_df)} valid records retained")

Entropy

In [ ]:
# Define base path
base_path = r'..\code\\'

# Read original data
df = pd.read_csv(os.path.join(base_path, 'patent_score.csv'))

# Define output function
def export_columns(columns, filename):
    """Helper function: validate and export specified columns"""
    # Verify if columns exist
    missing_cols = [col for col in columns if col not in df.columns]
    if missing_cols:
        print(f"Warning: missing columns {missing_cols}, skipping generation of {filename}")
        return
    
    # Extract and save data
    df[columns].to_csv(os.path.join(base_path, filename), index=False)
    print(f"Successfully generated: {filename}")

# Output three files as required
export_columns(['P', 'pr', 'PR'], 'data_pr.csv')
export_columns(['P', 'au', 'AU'], 'data_au.csv')
export_columns(['P', 'hub', 'HUB'], 'data_hub.csv')

In [ ]:
# ==============================
# 0. Data preparation (example data, replace with real data)
# ==============================

# data = pd.read_csv("data_pr.csv")
# df = pd.DataFrame({
#     'A': data['P'],      # Patent attribute score (right-skewed distribution)
#     'B': data['pr'],      # Patent network relationship score
#     'C': data['PR']  # Comprehensive score (simulated correlation)
# })

# data = pd.read_csv("data_au.csv")
# df = pd.DataFrame({
#     'A': data['P'],      # Patent attribute score (right-skewed distribution)
#     'B': data['au'],      # Patent network relationship score
#     'C': data['AU']  # Comprehensive score (simulated correlation)
# })

data = pd.read_csv("data_hub.csv")
df = pd.DataFrame({
    'A': data['P'],      # Patent attribute score (right-skewed distribution)
    'B': data['hub'],      # Patent network relationship score
    'C': data['HUB']  # Comprehensive score (simulated correlation)
})


# Normalize to [0,1] interval
scaler = MinMaxScaler()
df[['A_norm', 'B_norm', 'C_norm']] = scaler.fit_transform(df[['A', 'B', 'C']])

# ==============================
# 1. Define core functions
# ==============================
def discretize_data(series, bins=10):
    """Discretize continuous data into specified number of bins"""
    return pd.cut(series, bins=bins, labels=np.arange(bins))

def calculate_mi(df_subset):
    """Calculate mutual information statistics"""
    # Discretize data (number of bins adjusted based on data size)
    A_discrete = discretize_data(df_subset['A_norm'], bins=10)
    B_discrete = discretize_data(df_subset['B_norm'], bins=10)
    C_discrete = discretize_data(df_subset['C_norm'], bins=10)
    
    # Calculate mutual information
    mi_AC = mutual_info_score(A_discrete, C_discrete)
    mi_BC = mutual_info_score(B_discrete, C_discrete)
    
    # Joint mutual information
    joint_AB = pd.concat([A_discrete.astype(str) + "_" + B_discrete.astype(str)], axis=1)
    mi_ABC = mutual_info_score(joint_AB[0], C_discrete)
    
    # Conditional mutual information
    cond_mi_B_given_A = mi_ABC - mi_AC
    cond_mi_A_given_B = mi_ABC - mi_BC
    
    # Synergy effect
    synergy = mi_ABC - (mi_AC + mi_BC)
    
    return {
        'I(A;C)': mi_AC,
        'I(B;C)': mi_BC,
        'I(A,B;C)': mi_ABC,
        'I(B;C|A)': cond_mi_B_given_A,
        'I(A;C|B)': cond_mi_A_given_B,
        'Synergy': synergy
    }

# ==============================
# 2. Calculate statistics by percentage
# ==============================
results = []

# Generate percentage sequence: 5%, 10%, ..., 100%
percentages = np.arange(5, 101, 5)

for p in percentages:
    # Select top p% independently for each metric
    # ----------------------------------
    # Calculate threshold values for each metric
    a_threshold = df['A'].quantile(1 - p/100)
    b_threshold = df['B'].quantile(1 - p/100)
    c_threshold = df['C'].quantile(1 - p/100)
    
    # Get top indices for each metric
    top_A = df[df['A'] >= a_threshold].index
    top_B = df[df['B'] >= b_threshold].index
    top_C = df[df['C'] >= c_threshold].index
    
    # Merge indices and remove duplicates
    combined_index = top_A.union(top_B).union(top_C)
    df_subset = df.loc[combined_index].copy()
    
    # Calculate statistics
    stats = calculate_mi(df_subset)
    
    # Record results
    result = {
        'Percentage': f"Top {p}%",
        'Sample_Size': len(df_subset),
        'Sample_Ratio': len(df_subset)/len(df)
        #'x' : p
    }
    result.update(stats)
    results.append(result)

# ==============================
# 3. Result saving and display
# ==============================
result_df = pd.DataFrame(results)
print("\nResult summary (by percentage):")
print(result_df.round(4))

# Save to CSV
result_df.to_csv('patent_information_analysis.csv', index=False)

In [ ]:
# Set Seaborn style for better looking charts
sns.set(style="whitegrid")

# Read CSV file (ensure file path is correct)
df = pd.read_csv('patent_information_analysis.csv')

# Assume first column as x-axis, last six columns as y-axis variables
x = df.iloc[:, 0]                # X-axis data (e.g., year or other identifier)
y_columns = df.columns[-6:]      # Last six variables' column names

# Create plotting window
plt.figure(figsize=(12, 8))

# Loop to plot line chart for each variable
for col in y_columns:
    plt.plot(x, df[col], marker='o', label=col)

# Set title and axis labels
plt.title('Line chart showing last six variables', fontsize=16)
plt.xlabel(df.columns[0], fontsize=14)
plt.ylabel('Value', fontsize=14)

# Show legend and add grid
plt.legend(title="Variables", fontsize=12)
plt.grid(True)

# Show chart
plt.show()

## Network robustness and vulnerability indicator calculation

In [ ]:
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
from igraph import Graph, VertexClustering
import igraph as ig
import numpy as np

In [ ]:
# ========================
# 1. Data loading and constructing deletion order dataset
# ========================

# Patent metric data file path (modify according to actual situation)
score_path = r"patent_lcc_dataset.csv"
df_score = pd.read_csv(score_path)

# Assume data contains columns: 'patent_id', 'P', 'pr', 'PR', 'au', 'AU', 'hub', 'HUB'
# Construct deletion order dataset: record ranking after sorting each metric from high to low score
# Note that in R code, order() sorts in ascending order, if higher score is more important, pay attention to sort order.
# Here assume higher score is better, so sort each metric in descending order
metrics = ['P', 'pr', 'PR', 'au', 'AU', 'hub', 'HUB']
delete_data = pd.DataFrame({'patent_id': df_score['patent_id']})
for m in metrics:
    # Use ranking function: ranking 1 corresponds to highest scoring patent
    # method='min' ensures ranking consistency when scores are equal
    delete_data[m] = df_score[m].rank(method='min', ascending=False).astype(int)

# If need to view sort results, can print delete_data.head()
print("Deletion order dataset (partial):")
print(delete_data.head())

In [ ]:
# ========================
# 2. Build network and define indicator functions
# ========================

# Read edge data file (format assumed to be two columns, each row represents a connection)
df_edge = pd.read_csv(edge_path)
# If column names in file are not 'source' and 'target', modify according to actual situation
# For example, assume first column is 'from', second column is 'to'
#df_edge.columns = ['source', 'target']

# If patent numbers may be non-consecutive (string or number), suggest taking patent list
# Merge all patent ids
patent_ids = pd.unique(df_score['patent_id'])
# Build dictionary: patent id -> index
id2index = {pid: i for i, pid in enumerate(patent_ids)}
# Map edge data to node indices; note to check if patents in edge data are all in df_score
edges = []
for idx, row in df_edge.iterrows():
    a = row[0]  # First column
    b = row[1]  # Second column
    # If a and b are both in dictionary, retain
    if a in id2index and b in id2index:
        edges.append((id2index[a], id2index[b]))
        
# Create undirected graph
graph_gc = Graph()
graph_gc.add_vertices(len(patent_ids))
graph_gc.add_edges(edges)
graph_gc.simplify()  # Remove multiple edges and self-loops

print("Network nodes built:", graph_gc.vcount(), "edges:", graph_gc.ecount())

# Calculate original network global efficiency function
def global_efficiency(g: Graph) -> float:
    n = g.vcount()
    if n <= 1:
        return 0.0
    sum_eff = 0.0
    # Can use get_all_shortest_paths, or use shortest_paths method
    # Here call igraph's built-in shortest_paths (note default returns math.inf if no path)
    sp = g.shortest_paths_dijkstra()
    # np.allclose used for numerical comparison, but here only consider math.inf handling
    for i in range(n):
        for j in range(n):
            if i != j:
                d = sp[i][j]
                if d == math.inf or d == 0:  # Cannot reach or self-loop (should not occur)
                    continue
                sum_eff += 1.0/d
    # Total n*(n-1) pairs of nodes
    return sum_eff / (n*(n-1))

# Calculate original graph global efficiency and number of nodes in giant component
orig_eff = global_efficiency(graph_gc)

# Use igraph's connected_components() to get list of connected components
components = graph_gc.clusters(mode="weak")
orig_gc_size = max(components.sizes())

print("Original network global efficiency:", orig_eff)
print("Original network giant component node count:", orig_gc_size)

In [ ]:
# ========================
# 3. Node deletion and calculation of g and μ indicators
# ========================

# Fixed deletion ratio step (e.g., delete 5% nodes each time), delete according to sorting in delete_data for each metric
frac_step = 0.01
n_total = len(delete_data)  # Should be consistent with number of nodes in graph
step_nodes = max(1, int(frac_step * n_total))  # Number of nodes to delete per step

# Establish dictionary for saving indicator results
# Format: results[metric name] = { 'removal_frac': [], 'g': [], 'mu': [] }
results = {m: {'removal_frac': [], 'g': [], 'mu': []} for m in metrics}

for m in metrics:
    # Sort by current metric. Note: sorting basis is delete_data[m] value, smaller values first (i.e., higher scores first)
    order_df = delete_data[['patent_id', m]].sort_values(by=m, ascending=True).reset_index(drop=True)
    n = len(order_df)
    
    # Steps for number of nodes to delete (e.g.: j represents deleting first j nodes)
    for j in range(step_nodes, n+1, step_nodes):
        # Select first j patent nodes from sorted sequence
        remove_ids = order_df.loc[:j-1, 'patent_id'].tolist()
        # Convert patent ids to corresponding node indices in graph
        remove_indices = [id2index[pid] for pid in remove_ids if pid in id2index]
        # Copy original graph (suggest copying then deleting nodes, not modifying original)
        g_temp = graph_gc.copy()
        # Delete nodes, igraph's delete_vertices requires passing list of node indices
        # Node indices will be rearranged after deletion
        g_temp.delete_vertices(remove_indices)
        
        # Calculate new graph giant connectivity coefficient: max connected component nodes / original network giant component nodes
        comps = g_temp.clusters(mode="weak")
        if comps:
            new_gc_size = max(comps.sizes())
        else:
            new_gc_size = 0
        g_ratio = new_gc_size / orig_gc_size
        
        # Calculate new graph global efficiency and efficiency decline ratio μ
        new_eff = global_efficiency(g_temp)
        mu = 1 - (new_eff / orig_eff) if orig_eff > 0 else None
        
        # Record results, x-axis uses deletion ratio (deleted nodes / total nodes)
        results[m]['removal_frac'].append(j / n_total)
        results[m]['g'].append(g_ratio)
        results[m]['mu'].append(mu)
        
    print(f"Metric {m} processing completed, calculated {len(results[m]['removal_frac'])} steps")

In [ ]:
# ========================
# 4. Plot graphics
# ========================

# When plotting, draw 3 groups of curves by group
# First group: P, pr, PR
# Second group: P, au, AU
# Third group: P, hub, HUB

import matplotlib.pyplot as plt

def plot_curves(metric_list, indicator='g'):
    """
    Plot curves for each metric
    indicator: 'g' or 'mu'
    """
    plt.figure(figsize=(8,6))
    for m in metric_list:
        plt.plot(results[m]['removal_frac'], results[m][indicator], marker='o', label=m)
    plt.xlabel("Deletion ratio (deleted nodes/total nodes)")
    if indicator=='g':
        plt.ylabel("Giant connectivity coefficient g")
    elif indicator=='mu':
        plt.ylabel("Efficiency decline ratio μ")
    plt.legend()
    plt.grid(True)
    plt.title("Indicator {} changes with deletion ratio".format(indicator))
    plt.tight_layout()
    plt.show()

# Plot first group: P, pr, PR
plot_curves(['P', 'pr', 'PR'], indicator='g')
plot_curves(['P', 'pr', 'PR'], indicator='mu')

# Plot second group: P, au, AU
plot_curves(['P', 'au', 'AU'], indicator='g')
plot_curves(['P', 'au', 'AU'], indicator='mu')

# Plot third group: P, hub, HUB
plot_curves(['P', 'hub', 'HUB'], indicator='g')
plot_curves(['P', 'hub', 'HUB'], indicator='mu')

In [ ]:
# Initialize data dictionary
g_data = {'removal_frac': results['P']['removal_frac']}
mu_data = {'removal_frac': results['P']['removal_frac']}

# Add each metric's data as columns
for m in metrics:
    g_data[m] = results[m]['g']
    mu_data[m] = results[m]['mu']

# Build two DataFrames
df_g = pd.DataFrame(g_data)
df_mu = pd.DataFrame(mu_data)

# View results (first few rows)
print("DataFrame for giant connectivity coefficient g:")
print(df_g.head())

print("\nDataFrame for efficiency decline ratio μ:")
print(df_mu.head())

In [ ]:
df_g.to_csv('06df_g.csv')
df_mu.to_csv('06df_mu.csv')